# Emergency Sequencer Final Demo

This notebook imports the final `EmergencySequencer` implementation directly
and runs three representative descent scenarios.

- Normal-altitude descent with parachute and low-risk airbag flow
- Low-altitude fast-track descent with secondary cushion deployment
- High-risk descent where the airbag is withheld and cushion-only mitigation remains


In [ ]:
from pathlib import Path
import sys
import pandas as pd

CWD = Path.cwd()
SRC_CANDIDATES = [
    CWD / "src",
    CWD / "drone" / "src",
    CWD.parent / "src",
    CWD.parent / "drone" / "src",
    CWD.parent.parent / "drone" / "src",
    CWD,
]
for candidate in SRC_CANDIDATES:
    if (candidate / "emergency_sequencer.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("Could not locate drone src directory containing emergency_sequencer.py")

from emergency_sequencer import (
    CargoState,
    DescentPhysics,
    EmergencySequencer,
    ImpactRiskAssessment,
)


In [ ]:
def run_scenario(name, altitude, vertical_speed, risk, radar_distance=None, steps=80):
    seq = EmergencySequencer()
    physics = DescentPhysics(
        initial_altitude=altitude,
        initial_vertical_speed=vertical_speed,
        dt=0.1,
    )

    history = []
    ctx = physics.snapshot()
    ctx.radar_distance = radar_distance
    ctx.rotor_safe = None
    ctx.cargo_state = CargoState.LIQUID_FULL_HEAVY
    ctx.impact_risk = risk

    result = seq.start(ctx)
    history.append(
        {
            "step": 0,
            "phase": result.phase.name,
            "altitude": ctx.altitude_agl,
            "vertical_speed": ctx.vertical_speed,
            "tti": ctx.time_to_impact,
            "risk": risk.impact_zone_label,
            "secondary_cushion_deploy": bool(result.commands.get("secondary_cushion_deploy", False)),
            "parachute_deploy": bool(result.commands.get("parachute_deploy", False)),
            "airbag_prefill": bool(result.commands.get("airbag_prefill", False)),
            "airbag_fire": bool(result.commands.get("airbag_fire", False)),
            "log": result.log,
        }
    )

    if result.commands.get("parachute_deploy"):
        physics.deploy_chute()

    for step in range(1, steps + 1):
        if physics.is_terminal or seq.is_terminal:
            break

        physics.step(mass_kg=130.0)
        ctx = physics.snapshot()
        ctx.radar_distance = min(ctx.altitude_agl, 10.0)
        ctx.rotor_safe = None
        ctx.cargo_state = CargoState.LIQUID_FULL_HEAVY
        ctx.impact_risk = risk

        result = seq.step(ctx)
        if result.commands.get("parachute_deploy"):
            physics.deploy_chute()

        history.append(
            {
                "step": step,
                "phase": result.phase.name,
                "altitude": ctx.altitude_agl,
                "vertical_speed": ctx.vertical_speed,
                "tti": ctx.time_to_impact,
                "risk": risk.impact_zone_label,
                "secondary_cushion_deploy": bool(result.commands.get("secondary_cushion_deploy", False)),
                "parachute_deploy": bool(result.commands.get("parachute_deploy", False)),
                "airbag_prefill": bool(result.commands.get("airbag_prefill", False)),
                "airbag_fire": bool(result.commands.get("airbag_fire", False)),
                "log": result.log,
            }
        )

    df = pd.DataFrame(history)
    print(f"=== {name} ===")
    display(df.tail(12))
    return df


In [ ]:
low_risk = ImpactRiskAssessment.low_risk(score=0.18, reason="demo low-risk")
high_risk = ImpactRiskAssessment.high_risk(score=0.82, reason="demo high-risk")

scenario_a = run_scenario(
    "Scenario A: normal altitude / low risk",
    altitude=80.0,
    vertical_speed=8.0,
    risk=low_risk,
)

scenario_b = run_scenario(
    "Scenario B: low altitude fast-track / low risk",
    altitude=4.5,
    vertical_speed=12.0,
    risk=low_risk,
)

scenario_c = run_scenario(
    "Scenario C: low altitude fast-track / high risk",
    altitude=4.5,
    vertical_speed=12.0,
    risk=high_risk,
)
